# Notebook for Library merging

In [6]:
# Imports
from itertools import product
from sklearn.model_selection import ParameterGrid
from Models.config_parser import load_config, load_dataset_config
from Models.dataset_and_models import Dataset, Model

In [7]:
# Load configurations
CONFIG = "../configs/config-test.yaml"

model_config   = load_config(CONFIG)        # {model_key: {param: [values]}}
dataset_config = load_dataset_config(CONFIG) # {dataset_key: {n_features: [...], n_samples: [...]}}

In [8]:
# All model × hyperparameter combinations
model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]

# All dataset × (n_features, n_samples) combinations
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]


In [9]:
# Full cross-product: one entry per benchmark run
benchmark_runs = list(product(model_runs, dataset_runs))
print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs = {len(benchmark_runs)} total runs")

# Load a single dataset variant
dataset_key, dataset_params = dataset_runs[0]
ds = Dataset[dataset_key.upper()].load_dataset(**dataset_params)
X, y = ds["X"], ds["y"]

1 model configs × 4 dataset configs = 4 total runs


### Iterate trough every combination and train the model for explanation

In [10]:
from tqdm import tqdm
from Benchmarking import BenchmarkRunner
from Benchmarking.backends import ShapModeAgnosticBackend, ShapIQModeAgnosticBackend, LightShapValueBackend, DalexBackend

runner = BenchmarkRunner(
    backends=[ShapModeAgnosticBackend, ShapIQModeAgnosticBackend, LightShapValueBackend, DalexBackend],
    output_csv="../Benchmarking/results.csv",
    n_background=100,
    n_eval=None,
)

# Only tree models are compatible with ShapTrueValueBackend (TreeExplainer)
tree_model_keys = ["random_forest", "decision_tree", "gradient_boosting"]

for dataset_key, dataset_params in tqdm(dataset_runs, desc="datasets"):
    dataset_enum = Dataset[dataset_key.upper()]
    ds = dataset_enum.load_dataset(**dataset_params)
    X, y = ds["X"], ds["y"]

    tree_runs = [(mk, mp) for mk, mp in model_runs if mk in tree_model_keys]

    for model_key, model_params in tqdm(tree_runs, leave=False, desc="models"):
        trainer = Model[model_key.upper()].get_model_with_params(dataset_enum, model_params)
        trainer.fit(X, y, task=ds["task"])

        runner.run(
            model=trainer.get_model(),
            X=X,
            run_meta={
                "dataset": dataset_key,
                "model": model_key,
                "n_features": dataset_params.get("n_features"),
                "n_samples": dataset_params.get("n_samples"),
            },
        )

print("Done. Results written to Benchmarking/results.csv")

datasets:   0%|          | 0/4 [00:00<?, ?it/s]

Permutation SHAP (exact)








datasets:  25%|██▌       | 1/4 [00:49<02:27, 49.32s/it]

Permutation SHAP (exact)









datasets:  50%|█████     | 2/4 [01:50<01:52, 56.18s/it]

Permutation SHAP (exact)










datasets:  75%|███████▌  | 3/4 [02:58<01:01, 61.71s/it]

Permutation SHAP (exact)













datasets: 100%|██████████| 4/4 [04:25<00:00, 66.37s/it]

Done. Results written to Benchmarking/results.csv


In [11]:
import pandas as pd

results = pd.read_csv("../Benchmarking/results.csv")
print(f"{len(results)} rows written")
results.head(10)

16 rows written


,dataset,model,n_features,n_samples,backend,library,computation_type,chosen_method,n_eval,runtime_s,mean_abs_diff,sign_agreement,mean_sample_rho,reference_backend
0,california_housing,random_forest,2,1000,shap_mode_agnostic,shap,approximation,TreeExplainer,900,0.0045,NaN,NaN,NaN,NaN
1,california_housing,random_forest,2,1000,shapiq_mode_agnostic,shapiq,approximation,TreeExplainer,900,0.4874,NaN,NaN,NaN,NaN
2,california_housing,random_forest,2,1000,light_shap_value,LightShap,approximation,NaN,900,0.5812,NaN,NaN,NaN,NaN
3,california_housing,random_forest,2,1000,dalex_shap,dalex,approximation,shap_random_path_B25,900,48.2053,NaN,NaN,NaN,NaN
4,california_housing,random_forest,3,1000,shap_mode_agnostic,shap,approximation,TreeExplainer,900,0.0027,NaN,NaN,NaN,NaN
5,california_housing,random_forest,3,1000,shapiq_mode_agnostic,shapiq,approximation,TreeExplainer,900,0.1004,NaN,NaN,NaN,NaN
6,california_housing,random_forest,3,1000,light_shap_value,LightShap,approximation,NaN,900,0.8375,NaN,NaN,NaN,NaN
7,california_housing,random_forest,3,1000,dalex_shap,dalex,approximation,shap_random_path_B25,900,60.0084,NaN,NaN,NaN,NaN
8,california_housing,random_forest,4,1000,shap_mode_agnostic,shap,approximation,TreeExplainer,900,0.0106,NaN,NaN,NaN,NaN
9,california_housing,random_forest,4,1000,shapiq_mode_agnostic,shapiq,approximation,TreeExplainer,900,0.0970,NaN,NaN,NaN,NaN
